<a href="https://colab.research.google.com/github/kamalrawat77/agentic-iam-lab/blob/main/week07-agentic-rag/Nugget034_Agentic_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nugget 034: Agentic RAG (Decision-Making Retrieval)

In [ ]:
!pip install -q google-genai
!pip install sentence-transformers
import json
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

In [6]:
from google import genai

def callGPT(prompt):
  client = genai.Client(api_key="APIKEY")
  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt
  )

  return response

In [7]:
def cleanse_response(response):
  clean_response=clean_response = response.text
  clean_response = clean_response.replace("```json", "")
  clean_response = clean_response.replace("```", "")
  clean_response = clean_response.strip()

  return clean_response

In [54]:
def return_json(clean_response,objname):
  responseObj = json.loads(clean_response)
  if not objname:
    return responseObj
  resJsonObj  = responseObj[objname]
  return resJsonObj



In [9]:
def save_investigation(memory, investigation):

    memory.append(investigation)

    return memory

In [10]:
def get_last_investigation(memory):

    if len(memory) == 0:
        return None

    return memory[-1]

Load investigation history

Copy data from github project data/investigation_memory.json

In [11]:
data=[
  {
    "investigation_id": 1,
    "date": "2026-06-19",
    "dormant_accounts": 70,
    "inactive_approvers": 35,
    "sla_compliance": 72,
    "root_cause": "Inactive Approvers",
    "confidence": "HIGH",
    "recommendation": "Enable approver monitoring"
  },
  {
    "investigation_id": 2,
    "date": "2026-06-20",
    "dormant_accounts": 75,
    "inactive_approvers": 25,
    "sla_compliance": 70,
    "root_cause": "Workflow Bottleneck",
    "confidence": "HIGH",
    "recommendation": "Optimize workflow routing"
  },
  {
    "investigation_id": 3,
    "date": "2026-06-21",
    "dormant_accounts": 65,
    "inactive_approvers": 25,
    "sla_compliance": 35,
    "root_cause": "Provisioning Delays",
    "confidence": "HIGH",
    "recommendation": "Increase connector capacity"
  },
  {
    "investigation_id": 4,
    "date": "2026-06-22",
    "dormant_accounts": 85,
    "inactive_approvers": 15,
    "sla_compliance": 72,
    "confidence": "HIGH",
    "root_cause": "Approval Queue Growth",
    "recommendation": "Reduce approval backlog"
  },
  {
    "investigation_id": 5,
    "date": "2026-06-23",
    "dormant_accounts": 115,
    "inactive_approvers": 115,
    "sla_compliance": 72,
    "confidence": "HIGH",
    "root_cause": "Escalation Failure",
    "recommendation": "Review escalation rules"
  },
  {
    "investigation_id": 6,
    "date": "2026-06-23",
    "dormant_accounts": 115,
    "inactive_approvers": 115,
    "sla_compliance": 72,
    "confidence": "HIGH",
    "root_cause": "Dormant Accounts",
    "recommendation": "Perform Access review for Dormant accounts"
  },
  {
    "investigation_id": 7,
    "date": "2026-06-23",
    "dormant_accounts": 115,
    "inactive_approvers": 115,
    "sla_compliance": 72,
    "confidence": "HIGH",
    "root_cause": "Escalation Failure",
    "recommendation": "Review escalation rules"
  },
  {
    "investigation_id": 8,
    "date": "2026-06-23",
    "dormant_accounts": 115,
    "inactive_approvers": 115,
    "sla_compliance": 12,
    "confidence": "HIGH",
    "root_cause": "Certification SLA Failure",
    "recommendation": "Review Certification rules"
  },
  {
    "investigation_id": 9,
    "date": "2026-06-23",
    "dormant_accounts": 115,
    "inactive_approvers": 115,
    "sla_compliance": 72,
    "confidence": "HIGH",
    "root_cause": "Escalation Failure",
    "recommendation": "Review escalation rules"
  },
  {
    "investigation_id": 10,
    "date": "2026-06-23",
    "dormant_accounts": 115,
    "inactive_approvers": 115,
    "sla_compliance": 72,
    "confidence": "HIGH",
    "root_cause": "Escalation Failure",
    "recommendation": "Review escalation rules"
  }
]

Load data to current notebook context

In [12]:
with open("investigation_memory.json", "w") as f:
    json.dump(data, f)

Load investigation history now

In [13]:
with open("investigation_memory.json") as f:
    investigations = json.load(f)

print(investigations)

[{'investigation_id': 1, 'date': '2026-06-19', 'dormant_accounts': 70, 'inactive_approvers': 35, 'sla_compliance': 72, 'root_cause': 'Inactive Approvers', 'confidence': 'HIGH', 'recommendation': 'Enable approver monitoring'}, {'investigation_id': 2, 'date': '2026-06-20', 'dormant_accounts': 75, 'inactive_approvers': 25, 'sla_compliance': 70, 'root_cause': 'Workflow Bottleneck', 'confidence': 'HIGH', 'recommendation': 'Optimize workflow routing'}, {'investigation_id': 3, 'date': '2026-06-21', 'dormant_accounts': 65, 'inactive_approvers': 25, 'sla_compliance': 35, 'root_cause': 'Provisioning Delays', 'confidence': 'HIGH', 'recommendation': 'Increase connector capacity'}, {'investigation_id': 4, 'date': '2026-06-22', 'dormant_accounts': 85, 'inactive_approvers': 15, 'sla_compliance': 72, 'confidence': 'HIGH', 'root_cause': 'Approval Queue Growth', 'recommendation': 'Reduce approval backlog'}, {'investigation_id': 5, 'date': '2026-06-23', 'dormant_accounts': 115, 'inactive_approvers': 115,

Create searchable text.

In [14]:
documents = []

for inv in investigations:

    documents.append(
        f"""
        Root Cause:
        {inv['root_cause']}

        Recommendation:
        {inv['recommendation']}
        """
    )

Generate Embeddings

In [15]:
embeddings = model.encode(documents)

Create retrieval function and test it

In [46]:
def retrieve(query):

    query_embedding = model.encode(query)

    scores = util.cos_sim(
        query_embedding,
        embeddings
    )

    best_index = scores.argmax()

    return documents[best_index]

Create Agent decision function

In [47]:
def should_search(question):

    keywords = [
        "before",
        "history",
        "similar",
        "past",
        "previous"
    ]

    question = question.lower()

    for keyword in keywords:

        if keyword in question:
            return True

    return False

In [50]:
print(should_search("Have we seen this before"))

True


Build routing

In [51]:
question = """
Have we seen approval delays before?
"""

In [52]:
if should_search(question):

    print("SEARCH")

else:

    print("NO SEARCH")

SEARCH


Upgrade using Gemini.

In [64]:
prompt = f"""
Determine whether this question
requires searching historical
investigations.

Question:

{question}

Return JSON only:

{{
  "search": true
}}
"""

In [57]:
response=callGPT(prompt)

In [60]:
retrievalDecision=return_json(cleanse_response(response),None)

In [61]:
print(retrievalDecision)

{'search': True}


Connect to retrieval.

In [62]:
if retrievalDecision['search']:

    context = retrieve(question)

else:

    context = ""

In [63]:
print(context)


        Root Cause:
        Approval Queue Growth

        Recommendation:
        Reduce approval backlog
        


In [65]:
prompt = f"""
Question:

{question}

Context:

{context}

Answer the question.
"""

In [66]:
print(callGPT(prompt).text)

Yes, the context indicates that approval delays have been seen before. The "Root Cause: Approval Queue Growth" directly points to a situation where approvals are being delayed, leading to an increase in the queue. The "Recommendation: Reduce approval backlog" further confirms that these delays have created a backlog that needs to be addressed.


Additional Exercise

In [67]:
tools = [
    "search_history",
    "department_breakdown",
    "dormant_accounts"
]

In [73]:
prompt = f"""
Choose the best tool.

Available Tools:

{tools}

Question:

{question}

Return JSON only.
"""

In [77]:
print(callGPT(prompt).text)

```json
{
  "tool": "search_history"
}
```
